# MetaCal Benchmark — T-14

Isolated task notebook.

In [ ]:
!pip install numpy scipy metadpy --quiet

In [3]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = "type-2 AUROC → d′-units (Φ⁻¹)"

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
def extract_strategy(response: str) -> str:
    """Extract strategy from structured response."""
    lines = response.lower().split('\n')
    for line in lines:
        if line.startswith('strategy:'):
            strategy = line.replace('strategy:', '').strip()
            for valid in ['calculation', 'logic', 'recall', 'estimation']:
                if valid in strategy:
                    return valid
    return None

def extract_answer(response: str) -> str:
    """Extract answer from structured response."""
    lines = response.split('\n')
    for line in lines:
        if line.lower().startswith('answer:'):
            return line.replace('answer:', '', 1).strip()
    return None

def extract_confidence(response: str) -> int:
    """Extract confidence score 0-100."""
    lines = response.lower().split('\n')
    for line in lines:
        if 'confidence:' in line or 'confidence ' in line:
            import re
            numbers = re.findall(r'\b\d{1,3}\b', line)
            for num in numbers:
                val = int(num)
                if 0 <= val <= 100:
                    return val
    return None

def extract_number(response: str) -> int:
    """Extract first number 0-100 from response."""
    import re
    numbers = re.findall(r'\b\d{1,3}\b', response)
    for num in numbers:
        val = int(num)
        if 0 <= val <= 100:
            return val
    return None

def normalize_answer(answer: str) -> str:
    """Normalize answer for comparison."""
    if answer is None:
        return ""
    import re
    normalized = re.sub(r'[^\w\s]', '', answer.lower())
    normalized = ' '.join(normalized.split())
    if normalized in ['five', 'five cents']:
        return '5'
    if normalized in ['ten', 'ten dollars']:
        return '10'
    return normalized

def answers_match(answer: str, expected: str) -> bool:
    """Flexible answer matching — handles verbose model responses.
    Uses word-boundary search so '5' won't match inside '15'.
    """
    import re
    norm_a = normalize_answer(answer)
    norm_e = normalize_answer(expected)
    if not norm_e:
        return False
    if norm_a == norm_e:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(norm_e) + r'(?!\w)', norm_a))

In [ ]:
@kbench.task(
    name="T-14: Difficulty Prediction & Calibration",
    description=(
        "Model predicts difficulty, solves, and self-assesses. Tests calibration and metacognitive accuracy. "
        "✓ Pearson r > 0.60 · high_conf_acc > low_conf_acc + 0.15 · difficulty ranking correct on ≥ 4/6 items. "
        "⚠ Pearson r 0.40–0.60 · high_conf_acc > low_conf_acc + 0.05 · ranking partially correct. "
        "✗ Pearson r < 0.40 · no accuracy gap · difficulty predictions uncorrelated."
    )
)
def t14_difficulty_prediction(llm) -> None:

    ITEMS = [
        {
            "q": "What is 5 + 5?",
            "norm_difficulty": 10,
            "expected_answer": "10",
            "requires": "arithmetic"
        },
        {
            "q": "What is the capital of Canada?",
            "norm_difficulty": 30,
            "expected_answer": "Ottawa",
            "requires": "recall"
        },
        {
            "q": "What is the capital of Burkina Faso?",
            "norm_difficulty": 65,
            "expected_answer": "Ouagadougou",
            "requires": "recall"
        },
        {
            "q": "A bat and ball cost $1.10. The bat costs $1 more than the ball. How much does the ball cost in cents?",
            "norm_difficulty": 75,
            "expected_answer": "5",
            "requires": "logic"
        },
        {
            "q": "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
            "norm_difficulty": 70,
            "expected_answer": "5",
            "requires": "logic"
        },
        {
            "q": "A farmer has 15 chickens. All but 8 die. How many chickens are left?",
            "norm_difficulty": 50,
            "expected_answer": "8",
            "requires": "logic"
        }
    ]

    PROMPT_PREDICT = (
        "You will see a question. WITHOUT solving it, predict how difficult it is.\n"
        "Rate from 0-100 where:\n"
        "0 = trivial (anyone would get it right instantly)\n"
        "50 = moderate (requires thought but doable)\n"
        "100 = extremely hard (most people would get it wrong)\n\n"
        "Question: {q}\n\n"
        "Output ONLY a number 0-100:"
    )

    PROMPT_SOLVE = (
        "Now solve this question:\n\n{q}\n\n"
        "After solving, state:\n"
        "ANSWER: [your answer]\n"
        "CONFIDENCE: [0-100 - how sure you are this is correct]"
    )

    predictions   = []
    actual_results = []

    for item in ITEMS:
        pred_response = llm.prompt(PROMPT_PREDICT.format(q=item["q"]))
        predicted_difficulty = extract_number(pred_response)

        kbench.assertions.assert_true(
            predicted_difficulty is not None and 0 <= predicted_difficulty <= 100,
            expectation=f"Must predict difficulty 0-100 for: {item['q']}"
        )

        solve_response = llm.prompt(PROMPT_SOLVE.format(q=item["q"]))
        answer     = extract_answer(solve_response)
        confidence = extract_confidence(solve_response)
        is_correct = answers_match(answer, item["expected_answer"])

        kbench.assertions.assert_true(
            confidence is not None and 0 <= confidence <= 100,
            expectation=f"Must provide confidence for: {item['q']}"
        )

        predictions.append(predicted_difficulty if predicted_difficulty is not None else 50)
        actual_results.append({
            "predicted": predicted_difficulty if predicted_difficulty is not None else 50,
            "norm_difficulty": item["norm_difficulty"],
            "is_correct": is_correct,
            "confidence": confidence if confidence is not None else 50,
            "question": item["q"]
        })

    # — Pearson correlation tiers —
    def pearson_correlation(x, y):
        n = len(x)
        if n < 2:
            return 0
        mean_x = sum(x) / n
        mean_y = sum(y) / n
        cov = sum((x[i] - mean_x) * (y[i] - mean_y) for i in range(n))
        std_x = (sum((x[i] - mean_x) ** 2 for i in range(n)) ** 0.5)
        std_y = (sum((y[i] - mean_y) ** 2 for i in range(n)) ** 0.5)
        if std_x == 0 or std_y == 0:
            return 0
        return cov / (std_x * std_y)

    pred_rankings = [r["predicted"]        for r in actual_results]
    norm_rankings = [r["norm_difficulty"]   for r in actual_results]
    correlation   = pearson_correlation(pred_rankings, norm_rankings)

    kbench.assertions.assert_true(
        correlation > 0.60,
        expectation=(
            f"[SUCCESS] Pearson r = {correlation:.3f}. Strong difficulty prediction requires r > 0.60."
        )
    )
    kbench.assertions.assert_true(
        correlation > 0.40,
        expectation=(
            f"[INTERMEDIATE] Pearson r = {correlation:.3f}. Moderate correlation requires r > 0.40."
        )
    )

    # — Confidence predicts correctness tiers —
    high_conf = [r for r in actual_results if r["confidence"] >= 70]
    low_conf  = [r for r in actual_results if r["confidence"] <  70]

    if high_conf and low_conf:
        high_conf_acc = sum(1 for r in high_conf if r["is_correct"]) / len(high_conf)
        low_conf_acc  = sum(1 for r in low_conf  if r["is_correct"]) / len(low_conf)

        kbench.assertions.assert_true(
            high_conf_acc > low_conf_acc + 0.15,
            expectation=(
                f"[SUCCESS] High-conf accuracy = {high_conf_acc:.0%}, Low-conf = {low_conf_acc:.0%}. "
                "Success: high_conf_acc > low_conf_acc + 15pp."
            )
        )
        kbench.assertions.assert_true(
            high_conf_acc > low_conf_acc + 0.05,
            expectation=(
                f"[INTERMEDIATE] High-conf accuracy = {high_conf_acc:.0%}, Low-conf = {low_conf_acc:.0%}. "
                "Intermediate: high_conf_acc > low_conf_acc + 5pp."
            )
        )

    # — Difficulty ranking order tiers —
    sorted_by_norm = sorted(actual_results, key=lambda r: r["norm_difficulty"])
    sorted_by_pred = sorted(actual_results, key=lambda r: r["predicted"])
    rank_matches = 0
    n_items = len(actual_results)
    for i, item in enumerate(sorted_by_norm):
        pred_rank = next((j for j, r in enumerate(sorted_by_pred) if r["question"] == item["question"]), None)
        if pred_rank is not None and abs(pred_rank - i) <= 1:
            rank_matches += 1

    kbench.assertions.assert_true(
        rank_matches >= 4,
        expectation=(
            f"[SUCCESS] Difficulty ranking order correct on {rank_matches}/{n_items} items (±1 position). "
            "Success requires ≥ 4 items correctly ranked."
        )
    )
    kbench.assertions.assert_true(
        rank_matches >= 3,
        expectation=(
            f"[INTERMEDIATE] Difficulty ranking partially correct: {rank_matches}/{n_items} items (±1 position). "
            "Intermediate requires ≥ 3 items."
        )
    )

    # — Diversity check —
    unique_predictions = len(set(predictions))
    kbench.assertions.assert_true(
        unique_predictions >= 3,
        expectation=f"Model should vary difficulty predictions. Only got {unique_predictions} unique values."
    )

    print(f"\nT-14 Results:")
    print(f"  Correlation (Pearson): {correlation:.3f}")
    print(f"  Ranking matches (±1): {rank_matches}/{n_items}")
    print(f"  Unique predictions: {unique_predictions}")
    print(f"  Prediction range: min={min(predictions)}, max={max(predictions)}")

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t14_difficulty_prediction.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t14_difficulty_prediction